# エンベディングモデル推論ベンチマーク
## ruri-v3-130m / Qwen3-Embedding-0.6B

**目的**: AWS EC2インスタンス選定のためのレイテンシ・スループット・メモリ計測

| チェック項目 | 内容 |
|---|---|
| モデル1 | `cl-nagoya/ruri-v3-130m`（BERTエンコーダ型・132M params） |
| モデル2 | `Qwen/Qwen3-Embedding-0.6B`（デコーダ型LLM・0.6B params） |
| 計測項目 | 単一クエリ/バッチ処理のレイテンシ、メモリ使用量、CPU/GPU使用率 |

---
## セットアップ確認

In [1]:
# ========================================
# Cell 1: 環境確認 & パッケージインストール
# ========================================
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# 必要パッケージ（未インストールの場合のみ実行）
required = [
    "torch",
    "transformers>=4.48.0",
    "sentence-transformers>=3.0.0",
    "psutil",
    "accelerate",
]

for pkg in required:
    try:
        __import__(pkg.split("[")[0].split(">=")[0].split("==")[0].replace("-","_"))
    except ImportError:
        print(f"Installing {pkg}...")
        pip_install(pkg)

print("✅ パッケージ確認完了")

Installing sentence-transformers>=3.0.0...
✅ パッケージ確認完了


In [2]:
# ========================================
# Cell 2: 実行環境の詳細確認
# ========================================
import torch
import psutil
import platform

print("=" * 55)
print("  実行環境サマリー")
print("=" * 55)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  CPU         : {platform.processor() or '(取得不可)'}")
print(f"  CPU コア    : {psutil.cpu_count(logical=False)} 物理 / {psutil.cpu_count()} 論理")
total_ram = psutil.virtual_memory().total / (1024**3)
avail_ram = psutil.virtual_memory().available / (1024**3)
print(f"  RAM         : {total_ram:.1f} GB 合計 / {avail_ram:.1f} GB 空き")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    vram_free = (torch.cuda.get_device_properties(0).total_memory
                 - torch.cuda.memory_allocated(0)) / (1024**3)
    print(f"  GPU         : {gpu_name}")
    print(f"  VRAM        : {vram_total:.1f} GB 合計 / {vram_free:.1f} GB 空き")
    DEVICE = "cuda"
else:
    print("  GPU         : なし（CPU推論モード）")
    DEVICE = "cpu"

print(f"  推論デバイス: {DEVICE.upper()}")
print("=" * 55)

  実行環境サマリー
  Python      : 3.12.9
  PyTorch     : 2.6.0
  CPU         : x86_64
  CPU コア    : 2 物理 / 4 論理
  RAM         : 15.4 GB 合計 / 11.5 GB 空き
  GPU         : Tesla T4
  VRAM        : 14.6 GB 合計 / 14.6 GB 空き
  推論デバイス: CUDA


In [3]:
# ========================================
# Cell 3: ベンチマーク用サンプルテキスト
# ========================================

# 技術帳票・製品マニュアルを想定した日本語+英語混在サンプル
SAMPLE_TEXTS = {
    "short_ja": "エアコンのフィルターを月1回清掃してください。",  # ~18 tokens
    "short_en": "Please clean the air conditioner filter once a month.",
    "medium_ja": (
        "本製品はRoHS指令（2011/65/EU）に準拠しており、有害物質（鉛・水銀・カドミウム・"
        "六価クロム・ポリ臭化ビフェニル・ポリ臭化ジフェニルエーテル）の含有量が規制値以下"
        "であることを証明します。製品の廃棄は各国の法令に従い適切に行ってください。"
    ),  # ~80 tokens
    "mixed_ja_en": (
        "本モジュールはIEC 60068-2-6に準拠した振動試験（10〜500Hz、2G）および"
        "IEC 60068-2-27に準拠した衝撃試験（15G、11ms、半正弦波）をクリアしています。"
        "Operating temperature range: -10°C to +55°C, Storage: -20°C to +70°C."
        "IP保護等級はIP54（IEC 60529準拠）。"
    ),  # ~100 tokens, 混在
    "long_ja": (
        "第3章 保守・点検手順\n"
        "3.1 定期点検スケジュール\n"
        "本装置の安定稼働を維持するため、以下のスケジュールに従い定期点検を実施してください。\n"
        "日常点検（毎日）：運転状態表示ランプの確認、異常音・振動の有無確認、"
        "冷却ファンの動作確認。\n"
        "月次点検（毎月1回）：エアフィルター清掃・交換、端子部の増し締め、"
        "絶縁抵抗測定（500VDC、10MΩ以上）。\n"
        "年次点検（年1回）：電解コンデンサの容量測定・交換判定（定格容量の85%以下で交換）、"
        "バッテリーバックアップ回路の動作確認、全端子・コネクタの接触抵抗測定。\n"
        "3.2 フィルター交換手順\n"
        "1. 電源をOFFにし、主回路電源遮断後5分以上待機する（残留電荷の放電）。\n"
        "2. 前面パネルのネジ（M3×4本）を取り外す。\n"
        "3. フィルターユニット（品番：FU-3200-A）を前方へ引き出す。\n"
        "4. 新しいフィルターと交換後、逆手順で組み付ける。\n"
    ),  # ~250 tokens
}

# バッチテスト用（RAGのチャンク処理を模擬）
BATCH_TEXTS = [SAMPLE_TEXTS["medium_ja"]] * 5 + [SAMPLE_TEXTS["mixed_ja_en"]] * 5

print("✅ サンプルテキスト準備完了")
print(f"   - バッチサイズ: {len(BATCH_TEXTS)} テキスト")

✅ サンプルテキスト準備完了
   - バッチサイズ: 10 テキスト


In [4]:
# ========================================
# Cell 4: ベンチマーク計測ユーティリティ
# ========================================
import time
import gc
import numpy as np

def get_memory_usage_mb():
    """現在のプロセスのRAM使用量 (MB)"""
    proc = psutil.Process()
    return proc.memory_info().rss / (1024 ** 2)

def get_gpu_memory_mb():
    """GPU VRAM使用量 (MB)"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def benchmark_latency(encode_fn, texts, n_warmup=3, n_trials=10):
    """
    レイテンシベンチマーク
    - n_warmup: ウォームアップ実行回数（JITコンパイル等を安定させる）
    - n_trials: 計測回数
    Returns: dict{mean_ms, std_ms, p50_ms, p95_ms, min_ms, max_ms}
    """
    # ウォームアップ
    for _ in range(n_warmup):
        _ = encode_fn(texts)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    latencies = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        result = encode_fn(texts)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)  # ms

    arr = np.array(latencies)
    return {
        "mean_ms"  : float(arr.mean()),
        "std_ms"   : float(arr.std()),
        "p50_ms"   : float(np.percentile(arr, 50)),
        "p95_ms"   : float(np.percentile(arr, 95)),
        "min_ms"   : float(arr.min()),
        "max_ms"   : float(arr.max()),
    }

def print_benchmark_result(model_name, scenario, result, n_texts=1):
    tps = (n_texts / result["mean_ms"]) * 1000
    print(f"\n  [{model_name}] {scenario}")
    print(f"    平均レイテンシ : {result['mean_ms']:.1f} ms  (±{result['std_ms']:.1f})")
    print(f"    P50 / P95     : {result['p50_ms']:.1f} ms / {result['p95_ms']:.1f} ms")
    print(f"    Min / Max     : {result['min_ms']:.1f} ms / {result['max_ms']:.1f} ms")
    print(f"    スループット  : {tps:.1f} texts/sec")

print("✅ ユーティリティ関数 定義完了")

✅ ユーティリティ関数 定義完了


---
## Part 1: ruri-v3-130m
BERTエンコーダ型・132Mパラメータ・sentence-transformers互換

In [5]:
# ========================================
# Cell 5: ruri-v3-130m モデルロード
# ========================================
from sentence_transformers import SentenceTransformer

MODEL_ID_RURI = "cl-nagoya/ruri-v3-130m"

mem_before = get_memory_usage_mb()
gpu_before = get_gpu_memory_mb()

print(f"📥 モデルロード中: {MODEL_ID_RURI}")
print("   (初回実行時はHuggingFace Hubからダウンロードします。約520MB)")

t_load = time.perf_counter()
model_ruri = SentenceTransformer(
    MODEL_ID_RURI,
    device=DEVICE,
    trust_remote_code=False,   # ruri-v3はtrust不要
)
load_time = time.perf_counter() - t_load

mem_after = get_memory_usage_mb()
gpu_after = get_gpu_memory_mb()

print(f"\n✅ ロード完了 ({load_time:.1f}秒)")
print(f"   RAM増加量 : +{mem_after - mem_before:.0f} MB  (現在: {mem_after:.0f} MB)")
if DEVICE == "cuda":
    print(f"   VRAM増加量: +{gpu_after - gpu_before:.0f} MB  (現在: {gpu_after:.0f} MB)")
print(f"   最大シーケンス長: {model_ruri.max_seq_length} tokens")

2026-02-24 05:06:05.053471: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 05:06:05.223037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771909565.253643     471 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771909565.264440     471 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-24 05:06:05.454958: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

📥 モデルロード中: cl-nagoya/ruri-v3-130m
   (初回実行時はHuggingFace Hubからダウンロードします。約520MB)

✅ ロード完了 (6.7秒)
   RAM増加量 : +261 MB  (現在: 1348 MB)
   VRAM増加量: +515 MB  (現在: 515 MB)
   最大シーケンス長: 8192 tokens


In [6]:
# ========================================
# Cell 6: ruri-v3-130m 基本推論デモ
# ========================================
# ruri-v3はプレフィックスで検索クエリ/文書を区別する
# 詳細: https://huggingface.co/cl-nagoya/ruri-v3-130m

QUERY_PREFIX = "検索クエリ: "      # クエリ用プレフィックス
DOC_PREFIX   = "検索文書: "        # 文書（コーパス）用プレフィックス

# --- クエリ埋め込み ---
query = "エアコンフィルターの定期清掃方法"
query_emb = model_ruri.encode(
    QUERY_PREFIX + query,
    normalize_embeddings=True,   # コサイン類似度計算のために正規化
    show_progress_bar=False,
)

# --- 文書埋め込み（RAGのチャンク）---
docs = [
    DOC_PREFIX + SAMPLE_TEXTS["short_ja"],
    DOC_PREFIX + SAMPLE_TEXTS["medium_ja"],
    DOC_PREFIX + SAMPLE_TEXTS["long_ja"],
]
doc_embs = model_ruri.encode(
    docs,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=False,
)

# --- コサイン類似度計算 ---
from sentence_transformers import util as st_util
scores = st_util.cos_sim(query_emb, doc_embs)[0]

print("=" * 55)
print("  ruri-v3-130m 検索デモ")
print("=" * 55)
print(f"  クエリ: 「{query}」")
print(f"  ベクトル次元数: {query_emb.shape[0]}")
print()
for i, (doc, score) in enumerate(zip(docs, scores), 1):
    preview = doc.replace(DOC_PREFIX, "")[:35].replace("\n"," ")
    print(f"  Doc{i}: {preview}...")
    print(f"        類似度スコア: {score:.4f}")
print("=" * 55)

  ruri-v3-130m 検索デモ
  クエリ: 「エアコンフィルターの定期清掃方法」
  ベクトル次元数: 512

  Doc1: エアコンのフィルターを月1回清掃してください。...
        類似度スコア: 0.9253
  Doc2: 本製品はRoHS指令（2011/65/EU）に準拠しており、有害物質（...
        類似度スコア: 0.7563
  Doc3: 第3章 保守・点検手順 3.1 定期点検スケジュール 本装置の安定稼働...
        類似度スコア: 0.8763


In [7]:
# ========================================
# Cell 7: ruri-v3-130m レイテンシベンチマーク
# ========================================
print("=" * 55)
print("  ruri-v3-130m ベンチマーク開始")
print("=" * 55)

ruri_results = {}

# シナリオ1: 単一短文クエリ（リアルタイムRAGを想定）
def ruri_encode_single_query(texts):
    return model_ruri.encode(
        [QUERY_PREFIX + texts[0]],
        normalize_embeddings=True, show_progress_bar=False
    )

ruri_results["single_short"] = benchmark_latency(
    ruri_encode_single_query,
    [SAMPLE_TEXTS["short_ja"]]
)
print_benchmark_result("ruri-v3-130m", "単一クエリ（短文 ~18 tokens）",
                       ruri_results["single_short"], n_texts=1)

# シナリオ2: 単一長文ドキュメント埋め込み
def ruri_encode_single_doc(texts):
    return model_ruri.encode(
        [DOC_PREFIX + texts[0]],
        normalize_embeddings=True, show_progress_bar=False
    )

ruri_results["single_long"] = benchmark_latency(
    ruri_encode_single_doc,
    [SAMPLE_TEXTS["long_ja"]]
)
print_benchmark_result("ruri-v3-130m", "単一ドキュメント（長文 ~250 tokens）",
                       ruri_results["single_long"], n_texts=1)

# シナリオ3: バッチ処理（インデックス作成を想定）
BATCH_SIZE = 32
docs_with_prefix = [DOC_PREFIX + t for t in BATCH_TEXTS]

def ruri_encode_batch(texts):
    return model_ruri.encode(
        texts,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE,
        show_progress_bar=False
    )

ruri_results["batch"] = benchmark_latency(
    ruri_encode_batch,
    docs_with_prefix,
    n_trials=5
)
print_benchmark_result("ruri-v3-130m", f"バッチ処理（{len(BATCH_TEXTS)}テキスト / batch_size={BATCH_SIZE}）",
                       ruri_results["batch"], n_texts=len(BATCH_TEXTS))

print()
print(f"  メモリ使用量: RAM {get_memory_usage_mb():.0f} MB", end="")
if DEVICE == "cuda":
    print(f" / VRAM {get_gpu_memory_mb():.0f} MB")
else:
    print()

  ruri-v3-130m ベンチマーク開始

  [ruri-v3-130m] 単一クエリ（短文 ~18 tokens）
    平均レイテンシ : 16.3 ms  (±0.3)
    P50 / P95     : 16.3 ms / 16.8 ms
    Min / Max     : 15.9 ms / 16.8 ms
    スループット  : 61.4 texts/sec

  [ruri-v3-130m] 単一ドキュメント（長文 ~250 tokens）
    平均レイテンシ : 17.5 ms  (±0.6)
    P50 / P95     : 17.2 ms / 18.7 ms
    Min / Max     : 17.0 ms / 18.9 ms
    スループット  : 57.1 texts/sec

  [ruri-v3-130m] バッチ処理（10テキスト / batch_size=32）
    平均レイテンシ : 59.1 ms  (±0.7)
    P50 / P95     : 59.0 ms / 60.1 ms
    Min / Max     : 58.1 ms / 60.2 ms
    スループット  : 169.2 texts/sec

  メモリ使用量: RAM 1647 MB / VRAM 523 MB


---
## Part 2: Qwen3-Embedding-0.6B
デコーダ型LLMベース・0.6Bパラメータ・32Kトークン対応

In [8]:
# ========================================
# Cell 8: Qwen3-Embedding モデルロード
# ========================================
# Qwen3-EmbeddingはデコーダLLMベースのため、
# transformers直接使用 または sentence-transformers (>=3.0) で利用可能
#
# ⚠️ 注意: Qwen3-Embeddingは0.6Bでも約1.2GBのモデルファイル
#          GPU推奨 (VRAM >= 3GB)
#          CPUでも動作するが速度は大幅に低下

# ruri解放してメモリを確保
del model_ruri
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MODEL_ID_QWEN = "Qwen/Qwen3-Embedding-0.6B"

mem_before = get_memory_usage_mb()
gpu_before = get_gpu_memory_mb()

print(f"📥 モデルロード中: {MODEL_ID_QWEN}")
print("   (初回実行時はHuggingFace Hubからダウンロードします。約1.2GB)")

t_load = time.perf_counter()

# sentence-transformers経由（推奨: シンプルで互換性が高い）
model_qwen = SentenceTransformer(
    MODEL_ID_QWEN,
    device=DEVICE,
    model_kwargs={
        "torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32,
    },
    trust_remote_code=False,
)

load_time = time.perf_counter() - t_load
mem_after = get_memory_usage_mb()
gpu_after = get_gpu_memory_mb()

print(f"\n✅ ロード完了 ({load_time:.1f}秒)")
print(f"   RAM増加量 : +{mem_after - mem_before:.0f} MB  (現在: {mem_after:.0f} MB)")
if DEVICE == "cuda":
    print(f"   VRAM増加量: +{gpu_after - gpu_before:.0f} MB  (現在: {gpu_after:.0f} MB)")
print(f"   最大シーケンス長: {model_qwen.max_seq_length} tokens")

📥 モデルロード中: Qwen/Qwen3-Embedding-0.6B
   (初回実行時はHuggingFace Hubからダウンロードします。約1.2GB)


`torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]


✅ ロード完了 (12.8秒)
   RAM増加量 : +906 MB  (現在: 2553 MB)
   VRAM増加量: +1136 MB  (現在: 1144 MB)
   最大シーケンス長: 32768 tokens


In [9]:
# ========================================
# Cell 9: Qwen3-Embedding 基本推論デモ
# ========================================
# Qwen3-Embeddingはtask instructionをプロンプトとして受け取る
# クエリ: promptを指定 / 文書: promptなし（空文字列）
# 参考: https://huggingface.co/Qwen/Qwen3-Embedding

QWEN_QUERY_INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
# ※ RAGの用途に応じて変更してください:
# 技術文書: "Given a technical document query, retrieve relevant technical passages"
# 製品マニュアル: "Given a product manual query, retrieve relevant instructions"
# 日本語指定も可能: "質問に関連する技術文書の段落を検索してください"

query = "エアコンフィルターの定期清掃方法"

# クエリ埋め込み（promptを指定）
query_emb_q = model_qwen.encode(
    query,
    prompt=f"Instruct: {QWEN_QUERY_INSTRUCTION}\nQuery: ",
    normalize_embeddings=True,
    show_progress_bar=False,
)

# 文書埋め込み（promptなし）
docs_plain = [
    SAMPLE_TEXTS["short_ja"],
    SAMPLE_TEXTS["medium_ja"],
    SAMPLE_TEXTS["long_ja"],
]
doc_embs_q = model_qwen.encode(
    docs_plain,
    normalize_embeddings=True,
    batch_size=8,   # Qwen3はメモリ消費が大きいため小バッチ推奨
    show_progress_bar=False,
)

scores_q = st_util.cos_sim(query_emb_q, doc_embs_q)[0]

print("=" * 55)
print("  Qwen3-Embedding-0.6B 検索デモ")
print("=" * 55)
print(f"  クエリ: 「{query}」")
print(f"  ベクトル次元数: {query_emb_q.shape[0]}")
print()
for i, (doc, score) in enumerate(zip(docs_plain, scores_q), 1):
    preview = doc[:35].replace("\n"," ")
    print(f"  Doc{i}: {preview}...")
    print(f"        類似度スコア: {score:.4f}")
print("=" * 55)

  Qwen3-Embedding-0.6B 検索デモ
  クエリ: 「エアコンフィルターの定期清掃方法」
  ベクトル次元数: 1024

  Doc1: エアコンのフィルターを月1回清掃してください。...
        類似度スコア: 0.7793
  Doc2: 本製品はRoHS指令（2011/65/EU）に準拠しており、有害物質（...
        類似度スコア: 0.1475
  Doc3: 第3章 保守・点検手順 3.1 定期点検スケジュール 本装置の安定稼働...
        類似度スコア: 0.5625


In [10]:
# ========================================
# Cell 10: Qwen3-Embedding レイテンシベンチマーク
# ========================================
print("=" * 55)
print("  Qwen3-Embedding-0.6B ベンチマーク開始")
print("=" * 55)

qwen_results = {}
QWEN_PROMPT = f"Instruct: {QWEN_QUERY_INSTRUCTION}\nQuery: "

# シナリオ1: 単一短文クエリ
def qwen_encode_single_query(texts):
    return model_qwen.encode(
        [texts[0]],
        prompt=QWEN_PROMPT,
        normalize_embeddings=True,
        show_progress_bar=False
    )

qwen_results["single_short"] = benchmark_latency(
    qwen_encode_single_query,
    [SAMPLE_TEXTS["short_ja"]]
)
print_benchmark_result("Qwen3-Embedding-0.6B", "単一クエリ（短文 ~18 tokens）",
                       qwen_results["single_short"], n_texts=1)

# シナリオ2: 単一長文ドキュメント埋め込み
def qwen_encode_single_doc(texts):
    return model_qwen.encode(
        [texts[0]],
        normalize_embeddings=True,
        show_progress_bar=False
    )

qwen_results["single_long"] = benchmark_latency(
    qwen_encode_single_doc,
    [SAMPLE_TEXTS["long_ja"]]
)
print_benchmark_result("Qwen3-Embedding-0.6B", "単一ドキュメント（長文 ~250 tokens）",
                       qwen_results["single_long"], n_texts=1)

# シナリオ3: バッチ処理
QWEN_BATCH_SIZE = 8  # Qwen3はメモリ消費大のため小さめに

def qwen_encode_batch(texts):
    return model_qwen.encode(
        texts,
        normalize_embeddings=True,
        batch_size=QWEN_BATCH_SIZE,
        show_progress_bar=False
    )

qwen_results["batch"] = benchmark_latency(
    qwen_encode_batch,
    BATCH_TEXTS,
    n_trials=5
)
print_benchmark_result("Qwen3-Embedding-0.6B", f"バッチ処理（{len(BATCH_TEXTS)}テキスト / batch_size={QWEN_BATCH_SIZE}）",
                       qwen_results["batch"], n_texts=len(BATCH_TEXTS))

print()
print(f"  メモリ使用量: RAM {get_memory_usage_mb():.0f} MB", end="")
if DEVICE == "cuda":
    print(f" / VRAM {get_gpu_memory_mb():.0f} MB")
else:
    print()

  Qwen3-Embedding-0.6B ベンチマーク開始

  [Qwen3-Embedding-0.6B] 単一クエリ（短文 ~18 tokens）
    平均レイテンシ : 39.1 ms  (±0.4)
    P50 / P95     : 39.0 ms / 39.8 ms
    Min / Max     : 38.7 ms / 40.0 ms
    スループット  : 25.5 texts/sec

  [Qwen3-Embedding-0.6B] 単一ドキュメント（長文 ~250 tokens）
    平均レイテンシ : 48.1 ms  (±0.5)
    P50 / P95     : 48.0 ms / 48.8 ms
    Min / Max     : 47.4 ms / 49.0 ms
    スループット  : 20.8 texts/sec

  [Qwen3-Embedding-0.6B] バッチ処理（10テキスト / batch_size=8）
    平均レイテンシ : 121.9 ms  (±0.8)
    P50 / P95     : 121.6 ms / 123.1 ms
    Min / Max     : 121.3 ms / 123.4 ms
    スループット  : 82.1 texts/sec

  メモリ使用量: RAM 2637 MB / VRAM 1144 MB


---
## Part 3: 比較サマリー
計測結果をまとめてインスタンス選定の判断材料にする

In [11]:
# ========================================
# Cell 11: 比較サマリー出力
# ========================================
# ※ ruri_results と qwen_results の両方を実行済みの場合に動作します

scenarios = [
    ("単一クエリ（短文）",     "single_short", 1),
    ("単一ドキュメント（長文）","single_long",  1),
    ("バッチ10件",             "batch",        10),
]

print()
print("=" * 65)
print(f"  比較サマリー（デバイス: {DEVICE.upper()}）")
print("=" * 65)
print(f"  {'シナリオ':<24} {'ruri-v3-130m':>14} {'Qwen3-0.6B':>14} {'倍率':>6}")
print("-" * 65)

for label, key, n in scenarios:
    if key in ruri_results and key in qwen_results:
        r = ruri_results[key]["mean_ms"]
        q = qwen_results[key]["mean_ms"]
        ratio = q / r
        print(f"  {label:<24} {r:>10.1f} ms {q:>10.1f} ms {ratio:>5.1f}x")

print("=" * 65)
print()
print("  📌 インスタンス選定チェック")
print()

# 単一クエリのレイテンシからインスタンス判定
if "single_short" in ruri_results:
    r_lat = ruri_results["single_short"]["mean_ms"]
    if r_lat < 100:
        print(f"  ruri-v3-130m: {r_lat:.0f}ms → ✅ CPUインスタンスで十分です")
        print("     推奨: c7g.large ($52/月) または c6i.xlarge ($124/月)")
    elif r_lat < 500:
        print(f"  ruri-v3-130m: {r_lat:.0f}ms → ⚠️ CPU要件を確認してください")
        print("     推奨: c6i.2xlarge ($248/月) または g4dn.xlarge ($379/月)")
    else:
        print(f"  ruri-v3-130m: {r_lat:.0f}ms → ❌ より高性能なCPUかGPUが必要です")

if "single_short" in qwen_results:
    q_lat = qwen_results["single_short"]["mean_ms"]
    if q_lat < 200:
        print(f"  Qwen3-0.6B  : {q_lat:.0f}ms → ✅ 現インスタンスで実用的です")
    elif q_lat < 1000:
        print(f"  Qwen3-0.6B  : {q_lat:.0f}ms → ⚠️ GPUインスタンスを強く推奨します")
        print("     推奨: g4dn.xlarge ($379/月) または g5.xlarge ($734/月)")
    else:
        print(f"  Qwen3-0.6B  : {q_lat:.0f}ms → ❌ GPU必須 / CPUでは非実用的です")
        print("     推奨: g5.xlarge ($734/月)")

print("=" * 65)


  比較サマリー（デバイス: CUDA）
  シナリオ                       ruri-v3-130m     Qwen3-0.6B     倍率
-----------------------------------------------------------------
  単一クエリ（短文）                      16.3 ms       39.1 ms   2.4x
  単一ドキュメント（長文）                   17.5 ms       48.1 ms   2.7x
  バッチ10件                         59.1 ms      121.9 ms   2.1x

  📌 インスタンス選定チェック

  ruri-v3-130m: 16ms → ✅ CPUインスタンスで十分です
     推奨: c7g.large ($52/月) または c6i.xlarge ($124/月)
  Qwen3-0.6B  : 39ms → ✅ 現インスタンスで実用的です


In [12]:
# ========================================
# Cell 12: MRL（可変次元）デモ - Qwen3専用機能
# ========================================
# Qwen3-EmbeddingはMatryoshka Representation Learning対応
# ベクトル次元を小さくするとストレージ削減・検索高速化が可能
# デフォルト1024次元 → 256次元で品質を確認

print("  Qwen3-Embedding MRL（可変次元）デモ")
print("-" * 55)

query_test  = "エアコンフィルターの定期清掃方法"
doc_test    = SAMPLE_TEXTS["long_ja"]

for dim in [1024, 512, 256, 64]:
    q_emb = model_qwen.encode(
        query_test,
        prompt=QWEN_PROMPT,
        normalize_embeddings=True,
        truncate_dim=dim,       # MRL: 次元を切り詰める
    )
    d_emb = model_qwen.encode(
        doc_test,
        normalize_embeddings=True,
        truncate_dim=dim,
    )
    from numpy.linalg import norm
    score = float(np.dot(q_emb, d_emb) / (norm(q_emb) * norm(d_emb)))
    storage_per_vec_kb = (dim * 4) / 1024  # float32, KB
    print(f"  次元: {dim:4d} | 類似度: {score:.4f} | "
          f"ベクトルサイズ: {storage_per_vec_kb:.2f} KB")

print()
print("  ※ 512次元でも品質をほぼ維持できる場合が多い")
print("  ※ 100万ドキュメントの場合: 1024次元=4GB → 512次元=2GB")

  Qwen3-Embedding MRL（可変次元）デモ
-------------------------------------------------------
  次元: 1024 | 類似度: 0.5625 | ベクトルサイズ: 4.00 KB
  次元:  512 | 類似度: 0.6074 | ベクトルサイズ: 2.00 KB
  次元:  256 | 類似度: 0.6860 | ベクトルサイズ: 1.00 KB
  次元:   64 | 類似度: 0.7695 | ベクトルサイズ: 0.25 KB

  ※ 512次元でも品質をほぼ維持できる場合が多い
  ※ 100万ドキュメントの場合: 1024次元=4GB → 512次元=2GB


In [13]:
# ========================================
# Cell 13: 後処理 & クリーンアップ
# ========================================
print("クリーンアップ中...")
del model_qwen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"✅ 完了 / 現在のRAM使用量: {get_memory_usage_mb():.0f} MB")

print()
print("=" * 65)
print("  ベンチマーク完了")
print("=" * 65)
print("  次のステップ:")
print("  1. 各インスタンスタイプで本Notebookを実行し結果を比較")
print("     - CPU: c7g.large / c6i.xlarge")
print("     - GPU: g4dn.xlarge (T4) / g5.xlarge (A10G)")
print("  2. ruri-v3-130m が目標レイテンシを満たす最小インスタンスを選定")
print("  3. Qwen3-0.6B が必要な場合はGPUインスタンスのコスト対効果を判断")
print("=" * 65)

クリーンアップ中...
✅ 完了 / 現在のRAM使用量: 2637 MB

  ベンチマーク完了
  次のステップ:
  1. 各インスタンスタイプで本Notebookを実行し結果を比較
     - CPU: c7g.large / c6i.xlarge
     - GPU: g4dn.xlarge (T4) / g5.xlarge (A10G)
  2. ruri-v3-130m が目標レイテンシを満たす最小インスタンスを選定
  3. Qwen3-0.6B が必要な場合はGPUインスタンスのコスト対効果を判断
